# RiskLens: Previous applications

**Goal of this notebook:** `previous_application.csv` has the earlier loan applications each person made at Home Credit itself (not other banks, that was the bureau table). I want to turn it into a few features, one row per applicant, and see if they relate to default.

**Same problem as the bureau table:** many rows per applicant, so I summarise first and join after.

**What I'll do**
1. Look at the raw table and the status of past applications
2. Summarise per applicant (`groupby`)
3. Check the relation with `TARGET`

## 1. Loading the table and looking at the status 

In [4]:
import pandas as pd

path = "../data/raw/home-credit-default-risk/"
prev = pd.read_csv(path + "previous_application.csv")
print(prev.shape)
print(prev["NAME_CONTRACT_STATUS"].value_counts())

(1670214, 37)
NAME_CONTRACT_STATUS
Approved        1036781
Canceled         316319
Refused          290678
Unused offer      26436
Name: count, dtype: int64


The table has **1,670,214 rows and 37 columns**. Each row is one earlier application.

| Status | Count | Share |
|---|---|---|
| Approved | 1,036,781 | 62.1% |
| Canceled | 316,319 | 18.9% |
| Refused | 290,678 | 17.4% |
| Unused offer | 26,436 | 1.6% |

Most past applications were approved, but **17.4% were refused**. I find that interesting because a refusal is Home Credit's own earlier decision about the person, so it could say something about their risk. That's why I'm going to build a feature from it. I'm only counting "Refused" for now and leaving "Canceled" and "Unused offer" out, because it's not clear who cancelled or why.

## 2. One row per applicant

I grouped by `SK_ID_CURR` and made 4 features:

| Feature | What it is |
|---|---|
| `prev_count` | number of previous applications |
| `prev_refused` | how many of them were refused |
| `prev_refused_rate` | share of them that were refused |
| `prev_avg_credit` | average credit amount asked for |

In [5]:
prev["REFUSED"] = (prev["NAME_CONTRACT_STATUS"] == "Refused").astype(int)

prev_agg = prev.groupby("SK_ID_CURR").agg(
    prev_count=("SK_ID_PREV", "count"),
    prev_refused=("REFUSED", "sum"),
    prev_refused_rate=("REFUSED", "mean"),
    prev_avg_credit=("AMT_CREDIT", "mean")
).reset_index()

print(prev_agg.shape)
prev_agg.head()

(338857, 5)


,SK_ID_CURR,prev_count,prev_refused,prev_refused_rate,prev_avg_credit
0,100001,1,0,0.0,23787.00
1,100002,1,0,0.0,179055.00
2,100003,3,0,0.0,484191.00
3,100004,1,0,0.0,20106.00
4,100005,2,0,0.0,20076.75


Now there are **338,857 rows (one per applicant) and 5 columns** (the ID plus the 4 features). This includes applicants from the test set too, so not all of them are in my train table.

I added the refused *rate* on top of the refused count on purpose. Someone refused 2 times out of 2 is not the same as someone refused 2 times out of 10, and the count alone can't tell them apart.

Something to check later: `prev_avg_credit` is the average of `AMT_CREDIT`, and I haven't looked at what that column contains for refused or cancelled applications. If it's empty or zero for those, the average could be misleading.

## 3. Does it relate to default?

Same check as the bureau notebook: correlation of each new feature with `TARGET`.

In [7]:
app = pd.read_csv(path + "application_train.csv")
test = app[["SK_ID_CURR", "TARGET"]].merge(prev_agg, on="SK_ID_CURR", how="left")
print(test.drop(columns="SK_ID_CURR").corr()["TARGET"])
print("people with no previous applications:", test["prev_count"].isnull().sum())

TARGET               1.000000
prev_count           0.019762
prev_refused         0.064469
prev_refused_rate    0.077671
prev_avg_credit     -0.016114
Name: TARGET, dtype: float64
people with no previous applications: 16454


| Feature | Correlation with TARGET |
|---|---|
| prev_refused_rate | 0.078 |
| prev_refused | 0.064 |
| prev_count | 0.020 |
| prev_avg_credit | -0.016 |

- **`prev_refused_rate` is the best feature from this table** (0.078), and it's clearly more related to default than anything from the bureau table (the best there was 0.067). People who were refused before are more likely to default now.
- The rate works better than the raw count (0.078 vs 0.064), so building it was worth it.
- The number of previous applications (`prev_count`) and the average credit amount say almost nothing on their own.
- `SK_ID_CURR` also shows up in the output, but it's only an ID, so I ignore it.
- It's still a weak correlation (under 0.08), similar in size to `AGE` in the first notebook, so it won't be enough alone.
- **16454** applicants have no previous applications, they get NaN after the merge

## Summary

**What I found**
- 1.67 million previous applications, about 17% refused
- Grouped into one row per applicant (4 features)
- `prev_refused_rate` is the strongest new feature (0.078), better than the bureau features

**To check later**
- What `AMT_CREDIT` looks like for refused or canceled applications
- How many train applicants have no previous applications (I'll fill the counts with 0 in the merge notebook)

**Next:** merge everything into one table, add ratio features and save it